# Spotify Tracks - Meta Song Learner & Ensemble Clustering

This notebook demonstrates the implementation of the **Meta Song Learner** and **Ensemble (Consensus) Clustering**. 

The Meta Song Learner combines a supervised style classifier (External Service) with local, style-specific clustering models optimized via grid search. The Ensemble model combines predictions from K-Means, GMM, and Ward linkages into a single consensus partition.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, fcluster

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load Preprocessed Data

We load the same preprocessed sample generated during the EDA.

In [ ]:
data_path = "output/spotify_sampled_preprocessed.csv"
if not os.path.exists(data_path):
    data_path = "../output/spotify_sampled_preprocessed.csv"

if not os.path.exists(data_path):
    print("Error: Preprocessed dataset not found.")
else:
    df_sampled = pd.read_csv(data_path)
    features = [
        'popularity', 'duration_ms', 'danceability', 'energy', 'key', 
        'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 
        'liveness', 'valence', 'tempo', 'time_signature', 'explicit'
    ]
    df_sampled['explicit'] = df_sampled['explicit'].astype(float)
    X = df_sampled[features].values
    genres = df_sampled['track_genre'].values
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    print("Data loaded successfully. Shape:", X_scaled.shape)

## 2. Implement Genre Style Mapping

We map the 125 granular Spotify genres into 5 broad musical style categories.

In [ ]:
from meta_learner import map_genre_to_style

styles = np.array([map_genre_to_style(g) for g in genres])
print("Unique broad styles mapped:", np.unique(styles))
pd.Series(styles).value_counts()

## 3. Train the External Classification Service

We train and evaluate a supervised `RandomForestClassifier` to predict the song style from the audio features.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, styles, test_size=0.2, random_state=42, stratify=styles
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"Supervised Style Classifier Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

## 4. Fit the Meta Song Learner

We run the Meta Song Learner which fits the style classifier and fine-tunes a local clustering model (KMeans, GMM, or DBSCAN) for each style category.

In [ ]:
from meta_learner import MetaSongLearner

meta_learner = MetaSongLearner()
meta_learner.fit(X_scaled, genres)

print("\n--- Fine-Tuning Summary ---")
for style, model_info in meta_learner.style_models.items():
    print(f"Style: {style} => Best Model: {model_info[0]}")

## 5. Evaluate and Visualize Meta Learner Clusters

We obtain the global cluster labels and plot them using PCA.

In [ ]:
meta_labels = meta_learner.predict_song_cluster(X_scaled)
mask_meta = meta_labels != -1
num_meta_clusters = len(np.unique(meta_labels[mask_meta]))

meta_sil = silhouette_score(X_scaled[mask_meta], meta_labels[mask_meta])
meta_ch = calinski_harabasz_score(X_scaled[mask_meta], meta_labels[mask_meta])
meta_db = davies_bouldin_score(X_scaled[mask_meta], meta_labels[mask_meta])

print(f"Meta Learner Global Evaluation (K={num_meta_clusters}):")
print(f"  Silhouette Score: {meta_sil:.4f}")
print(f"  Calinski-Harabasz: {meta_ch:.2f}")
print(f"  Davies-Bouldin: {meta_db:.4f}")

# Plot PCA
pca_2d = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
colors_meta = plt.cm.get_cmap('tab20', max(20, num_meta_clusters))
scatter_colors_meta = [(0.7, 0.7, 0.7, 0.5) if l == -1 else colors_meta(l % 20) for l in meta_labels]

plt.figure(figsize=(10, 7))
plt.scatter(pca_2d[:, 0], pca_2d[:, 1], c=scatter_colors_meta, s=20, edgecolors='none')
plt.title(f'Meta Song Learner Clusters in PCA Space (K={num_meta_clusters})')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

## 6. Ensemble (Consensus) Clustering

We run the ensemble clustering combination which computes a co-association matrix across K-Means, GMM, and Ward linkages to form a consensus partition.

In [ ]:
from meta_learner import run_ensemble_clustering

ensemble_labels = run_ensemble_clustering(X_scaled, n_clusters=4)
ens_sil = silhouette_score(X_scaled, ensemble_labels)
ens_ch = calinski_harabasz_score(X_scaled, ensemble_labels)
ens_db = davies_bouldin_score(X_scaled, ensemble_labels)

print(f"\nEnsemble Clustering Global Evaluation (K=4):")
print(f"  Silhouette Score: {ens_sil:.4f}")
print(f"  Calinski-Harabasz: {ens_ch:.2f}")
print(f"  Davies-Bouldin: {ens_db:.4f}")

# Plot PCA
colors_ens = plt.cm.get_cmap('tab10', 4)
scatter_colors_ens = [colors_ens(l) for l in ensemble_labels]

plt.figure(figsize=(10, 7))
plt.scatter(pca_2d[:, 0], pca_2d[:, 1], c=scatter_colors_ens, s=20, edgecolors='none')
plt.title('Ensemble Consensus Clusters in PCA Space (K=4)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()